# Cleaning the Dataset & Feature Engineering

In [7]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('russell3000_quarterly_with_price_mapping.csv')

print("Current number of columnns pre feature engineering: ", df.columns.value_counts().sum())

print("Here are my columns I'm working with to create features from: ")
for i, col in enumerate(df.columns, start=1): # Looked up this function; allows me to get around the limitations of .columns not displaying every column name
    print(i, col)

Current number of columnns pre feature engineering:  407
Here are my columns I'm working with to create features from: 
1 Ticker
2 fiscal_year
3 fiscal_period
4 period_end_date
5 inc_currency
6 inc_revenue
7 inc_revenue_product
8 inc_revenue_services
9 inc_cost_of_revenue
10 inc_gross_profit
11 inc_operating_expenses
12 inc_research_development
13 inc_selling_general_administrative
14 inc_selling_marketing_expenses
15 inc_general_admin_expenses
16 inc_depreciation_amortization
17 inc_depreciation
18 inc_amortization_of_intangibles
19 inc_stock_based_compensation
20 inc_operating_income
21 inc_interest_expense
22 inc_interest_income
23 inc_other_income_expense
24 inc_income_before_taxes
25 inc_income_tax_expense
26 inc_net_income_continuing_operations
27 inc_net_income_discontinued_operations
28 inc_net_income
29 inc_basic_shares_outstanding
30 inc_diluted_shares_outstanding
31 inc_weighted_average_basic_shares
32 inc_weighted_average_diluted_shares
33 inc_basic_eps
34 inc_diluted_eps
3

In [8]:
# Feature Engineering Time!

# After having importanted 10-Q and 10-K data from StockAiInsight's subcription API and utilizing yfinance to obtain historical stock price data,
# I can now begin the process of feature engineering.

# Let's begin:

###################################################
# 0) Prerequisities for Smooth Feature Engineering
###################################################

# Make "Ticker" column name to "ticker" because it's annoying me
df.rename(columns={"Ticker": "ticker"}, inplace=True)

# To make sure date is a real date and data is sorted correctly
df["period_end_date"] = pd.to_datetime(df["period_end_date"])
df = df.sort_values(["ticker", "period_end_date"]).copy()

# Safe Division Function
def safe_div(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    result = numerator / denominator
    return result.replace([np.inf, -np.inf], np.nan)

# Safe Division Function for ratios that only make sense with a positive denominator
def safe_div_positive(numerator, denominator):
    denominator = denominator.where(denominator > 0, np.nan)
    return safe_div(numerator, denominator)


###################################################
# 1) Conventions and Building Blocks
###################################################

# Basic Market Cap Q
df['basic_market_cap_q'] = df['price_close_q'] * df['shr_basic_shares_outstanding']

# Dilutted Market Cap Q
df['diluted_market_cap_q'] = df['price_close_q'] * df['shr_diluted_shares_outstanding']

# Enterprise Value Q
df['enterprise_value_q'] = df['diluted_market_cap_q'] + df['bal_total_debt'] - df['bal_cash_and_equivalents']


###################################################
# 2) Average Balance Sheet Denominators
###################################################

# Average Total Assets Q
df['avg_total_assets_q'] = (
    df['bal_total_assets'] + df.groupby('ticker')['bal_total_assets'].shift(1)
) / 2

# Average Total Equity Q
df['avg_total_equity_q'] = (
    df['bal_total_equity'] + df.groupby('ticker')['bal_total_equity'].shift(1)
) / 2

# Average Total Liabilities Q
df['avg_total_liabilities_q'] = (
    df['bal_total_liabilities'] + df.groupby('ticker')['bal_total_liabilities'].shift(1)
) / 2

# Average Total Debt Q
df['avg_total_debt_q'] = (
    df['bal_total_debt'] + df.groupby('ticker')['bal_total_debt'].shift(1)
) / 2

# Average Accounts Receivable Q
df['avg_accounts_receivable_q'] = (
    df['bal_accounts_receivable'] + df.groupby('ticker')['bal_accounts_receivable'].shift(1)
) / 2

# Average Inventory Q
df['avg_inventory_q'] = (
    df['bal_inventory'] + df.groupby('ticker')['bal_inventory'].shift(1)
) / 2

# Average Accounts Payable Q
df['avg_accounts_payable_q'] = (
    df['bal_accounts_payable'] + df.groupby('ticker')['bal_accounts_payable'].shift(1)
) / 2

# Average Working Capital Q
df['avg_working_capital_q'] = (
    df['bal_working_capital'] + df.groupby('ticker')['bal_working_capital'].shift(1)
) / 2

# Average PPE Q
df['avg_ppe_q'] = (
    df['bal_property_plant_equipment'] + df.groupby('ticker')['bal_property_plant_equipment'].shift(1)
) / 2


###################################################
# 3) Revenue Mix and Composition
###################################################

# Product Revenue Share Q
df['product_revenue_share_q'] = safe_div(df['inc_revenue_product'], df['inc_revenue'])

# Services Revenue Share Q
df['services_revenue_share_q'] = safe_div(df['inc_revenue_services'], df['inc_revenue'])

# Contract Revenue excluding Tax Share Q
df['contract_revenue_excluding_tax_share_q'] = safe_div(df['inc_revenue_from_contracts_excluding_tax'], df['inc_revenue'])

# Contract Revenue including Tax Share Q
df['contract_revenue_including_tax_share_q'] = safe_div(df['inc_revenue_from_contracts_including_tax'], df['inc_revenue'])


###################################################
# 4) Core Profitability Ratios
###################################################

# Gross Margin Q
df['gross_margin_q'] = safe_div(df['inc_gross_profit'], df['inc_revenue'])

# Operating Margin Q
df['operating_margin_q'] = safe_div(df['inc_operating_income'], df['inc_revenue'])

# Net Margin Q
df['net_margin_q'] = safe_div(df['inc_net_income'], df['inc_revenue'])

# Pretax Margin Q
df['pretax_margin_q'] = safe_div(df['inc_income_before_taxes'], df['inc_revenue'])

# Common Earnings Margin Q
df['common_earnings_margin_q'] = safe_div(df['inc_net_income_available_to_common'], df['inc_revenue'])

# Continuing Operations Margin Q
df['continuing_operations_margin_q'] = safe_div(df['inc_net_income_continuing_operations'], df['inc_revenue'])

# R&D Intensity Q
df['R&D_intensity_q'] = safe_div(df['inc_research_development'], df['inc_revenue'])

# SG&A Intensity Q
df['SG&A_intensity_q'] = safe_div(df['inc_selling_general_administrative'], df['inc_revenue'])

# Marking Intensity Q
df['marketing_intensity_q'] = safe_div(df['inc_selling_marketing_expenses'], df['inc_revenue'])

# G&A Intensity Q
df['G&A_intensity_q'] = safe_div(df['inc_general_admin_expenses'], df['inc_revenue'])

# D&A Intensity Q
df['D&A_intensity_q'] = safe_div(df['inc_depreciation_amortization'], df['inc_revenue'])

# Depreciation Intensity Q
df['depreciation_intensity_q'] = safe_div(df['inc_depreciation'], df['inc_revenue'])

# Amortization Intensity Q
df['amortization_intensity_q'] = safe_div(df['inc_amortization_of_intangibles'], df['inc_revenue'])

# Stock Comp Intensity Q
df['stock_comp_intensity_q'] = safe_div(df['inc_stock_based_compensation'], df['inc_revenue'])

# Impairment Intensity Q
df['impairment_intensity_q'] = safe_div(df['inc_impairment_charges'], df['inc_revenue'])

# Restructuring Intensity Q
df['restructuring_intensity_q'] = safe_div(df['inc_restructuring_costs'], df['inc_revenue'])

# Comprehensive Income Margin Q
df['comprehensive_income_margin_q'] = safe_div(df['inc_comprehensive_income'], df['inc_revenue'])

# Tax Burden Ratio Q
df['tax_burden_ratio_q'] = safe_div_positive(df['inc_net_income'], df['inc_income_before_taxes'])

# Effective Rate Q
df['effective_tax_rate_q'] = safe_div_positive(df['inc_income_tax_expense'], df['inc_income_before_taxes'])


###################################################
# 5) Return Ratios
###################################################

# ROA Q
df['ROA_q'] = safe_div_positive(df['inc_net_income'], df['avg_total_assets_q'])

# ROE Q
df['ROE_q'] = safe_div_positive(df['inc_net_income_available_to_common'], df['avg_total_equity_q'])

# Operating ROA Q
df['operating_ROA_q'] = safe_div_positive(df['inc_operating_income'], df['avg_total_assets_q'])

# Pretax ROA Q
df['pretax_ROA_q'] = safe_div_positive(df['inc_income_before_taxes'], df['avg_total_assets_q'])

# Cash ROA Q
df['cash_ROA_q'] = safe_div_positive(df['cf_net_cash_from_operations'], df['avg_total_assets_q'])

# FCF ROA Q
df['FCF_ROA_q'] = safe_div_positive(df['cf_free_cash_flow'], df['avg_total_assets_q'])


###################################################
# 6) Liquidity Ratios
###################################################

# Current Ratio Q
df['current_ratio_q'] = safe_div_positive(df['bal_current_assets'], df['bal_current_liabilities'])

# Quick Ratio Q
df['quick_ratio_q'] = safe_div_positive(
    (df['bal_current_assets'] - df['bal_inventory'] - df['bal_prepaid_expenses']),
    df['bal_current_liabilities']
)

# Cash Ratio Q
df['cash_ratio_q'] = safe_div_positive(df['bal_cash_and_equivalents'], df['bal_current_liabilities'])

# Working Capital to Assets Q
df['working_capital_to_assets_q'] = safe_div_positive(df['bal_working_capital'], df['bal_total_assets'])

# Cash to Assets Q
df['cash_to_assets_q'] = safe_div_positive(df['bal_cash_and_equivalents'], df['bal_total_assets'])

# Cash & Shoret Term Investments to Assets Q
df['cash_and_short_term_investments_to_assets_q'] = safe_div_positive(
    (df['bal_cash_and_equivalents'] + df['bal_short_term_investments']),
    df['bal_total_assets']
)


###################################################
# 7) Leverage and Solvency Ratios
###################################################

# Debt to Equity Q
df['debt_to_equity_q'] = safe_div_positive(df['bal_total_debt'], df['bal_total_equity'])

# Debt to Assets Q
df['debt_to_assets_q'] = safe_div_positive(df['bal_total_debt'], df['bal_total_assets'])

# Liabilities to Assets Q
df['liabilities_to_assets_q'] = safe_div_positive(df['bal_total_liabilities'], df['bal_total_assets'])

# Net Debt to Assets Q
df['net_debt_to_assets_q'] = safe_div_positive(df['bal_net_debt'], df['bal_total_assets'])

# Net Debt to Equity Q
df['net_debt_to_equity_q'] = safe_div_positive(df['bal_net_debt'], df['bal_total_equity'])

# Long Term Debt to Capital Q
df['long_term_debt_to_capital_q'] = safe_div_positive(
    df['bal_long_term_debt'],
    (df['bal_long_term_debt'] + df['bal_total_equity'])
)

# Short Term Debt Share Q
df['short_term_debt_share_q'] = safe_div_positive(df['bal_short_term_debt'], df['bal_total_debt'])

# Long Term Debt Share Q
df['long_term_debt_share_q'] = safe_div_positive(df['bal_long_term_debt'], df['bal_total_debt'])

# Lease Liabilities to Assets Q
df['lease_liabilities_to_assets_q'] = safe_div_positive(
    (df['bal_operating_lease_liability_current'] + df['bal_operating_lease_liability_noncurrent']),
    df['bal_total_assets']
)

# Debt to Tangible Book Q
df['debt_to_tangible_book_q'] = safe_div_positive(df['bal_total_debt'], df['bal_tangible_book_value'])

# Cash to Debt Q
df['cash_to_debt_q'] = safe_div_positive(df['bal_cash_and_equivalents'], df['bal_total_debt'])


###################################################
# 8) Coverage Ratios
###################################################

# Interest Coverage Q
df['interest_coverage_q'] = safe_div_positive(df['inc_operating_income'], df['inc_interest_expense'])

# EBITDA Style Coverage Q
df['EBITDA_style_coverage_q'] = safe_div_positive(
    (df['inc_operating_income'] + df['inc_depreciation_amortization']),
    df['inc_interest_expense']
)

# Cash Flow Interest Coverage Q
df['cash_flow_interest_coverage_q'] = safe_div_positive(df['cf_net_cash_from_operations'], df['inc_interest_expense'])

# Debt Service Proxy Q
df['debt_service_proxy_q'] = safe_div_positive(
    df['cf_net_cash_from_operations'],
    (df['inc_interest_expense'] + df['cf_repayment_of_debt'])
)


###################################################
# 9) Efficiency and Turnover Ratios
###################################################

# Asset Turnover Q
df['asset_turnover_q'] = safe_div_positive(df['inc_revenue'], df['avg_total_assets_q'])

# Receivables Turnover Q
df['receivables_turnover_q'] = safe_div_positive(df['inc_revenue'], df['avg_accounts_receivable_q'])

# Inventory Turnover Q
df['inventory_turnover_q'] = safe_div_positive(df['inc_cost_of_revenue'], df['avg_inventory_q'])

# Payables Turnover Q
df['payables_turnover_q'] = safe_div_positive(df['inc_cost_of_revenue'], df['avg_accounts_payable_q'])

# PPE Turnover Q
df['ppe_turnover_q'] = safe_div_positive(df['inc_revenue'], df['avg_ppe_q'])

# Working Capital Turnover Q
df['working_capital_turnover_q'] = safe_div_positive(df['inc_revenue'], df['avg_working_capital_q'])


###################################################
# 10) Working Capital Cycle Features
###################################################

# Days Sales Outstanding Q
df['days_sales_outstanding_q'] = (
    safe_div_positive(df['bal_accounts_receivable'], df['inc_revenue'])
) * 90

# Days Inventory Outstanding Q
df['days_inventory_outstanding_q'] = (
    safe_div_positive(df['bal_inventory'], df['inc_cost_of_revenue'])
) * 90

# Days Payables Outstanding Q
df['days_payables_outstanding_q'] = (
    safe_div_positive(df['bal_accounts_payable'], df['inc_cost_of_revenue'])
) * 90

# Cash Conversion Cycle Q
df['cash_conversion_cycle_q'] = (
    df['days_sales_outstanding_q'] + df['days_inventory_outstanding_q'] - df['days_payables_outstanding_q']
)


###################################################
# 11) Cash Flow Quality and Strength Ratios
###################################################

# OFC Margin Q
df['OFC_margin_q'] = safe_div_positive(df['cf_net_cash_from_operations'], df['inc_revenue'])

# FCF Margin Q
df['FCF_margin_q'] = safe_div_positive(df['cf_free_cash_flow'], df['inc_revenue'])

# Levered FCF Margin Q
df['levered_FCF_margin_q'] = safe_div_positive(df['cf_levered_free_cash_flow'], df['inc_revenue'])

# OCF to Net Income Q
df['OCF_to_net_income_q'] = safe_div_positive(df['cf_net_cash_from_operations'], df['inc_net_income'])

# FCF to Net Income Q
df['FCF_to_net_income_q'] = safe_div_positive(df['cf_free_cash_flow'], df['inc_net_income'])

# OCF to Liabilities Q
df['OCF_to_liabilities_q'] = safe_div_positive(df['cf_net_cash_from_operations'], df['bal_total_liabilities'])

# FCF to Debt Q
df['FCF_to_debt_q'] = safe_div_positive(df['cf_free_cash_flow'], df['bal_total_debt'])

# Capex to Revenue Q
df['capex_to_revenue_q'] = safe_div_positive(df['cf_capital_expenditures'], df['inc_revenue'])

# Capex to Assets Q
df['capex_to_assets_q'] = safe_div_positive(df['cf_capital_expenditures'], df['avg_total_assets_q'])

# Recurring Capex to Revenue Q
df['recurring_capex_to_revenue_q'] = safe_div_positive(df['cf_recurring_capital_expenditures'], df['inc_revenue'])

# Acquisition Intensity Q
df['acquisition_intensity_q'] = safe_div_positive(df['cf_acquisitions'], df['inc_revenue'])

# Investment Purchase Intensity Q
df['investment_purchase_intensity_q'] = safe_div_positive(df['cf_purchase_of_investments'], df['bal_total_assets'])

# Investment Proceeds Intensity Q
df['investment_proceeds_intensity_q'] = safe_div_positive(df['cf_proceeds_from_investments'], df['bal_total_assets'])

# Cash Conversion Ratio Q
df['cash_conversion_ratio_q'] = safe_div_positive(df['cf_net_cash_from_operations'], df['inc_operating_income'])


###################################################
# 12) Capital Allocation and Payout Ratios
###################################################

# Buyback to Market Cap Q
df['buyback_to_market_cap_q'] = safe_div_positive(df['cf_repurchase_of_stock'], df['diluted_market_cap_q'])

# Buyback to FCF Q
# Only keep when free cash flow is positive so the ratio stays economically meaningful
df['buyback_to_FCF_q'] = safe_div_positive(df['cf_repurchase_of_stock'], df['cf_free_cash_flow'])

# Buyback to OCF Q
df['buyback_to_OCF_q'] = safe_div_positive(df['cf_repurchase_of_stock'], df['cf_net_cash_from_operations'])

# Buyback to Revenue Q
df['buyback_to_revenue_q'] = safe_div_positive(df['cf_repurchase_of_stock'], df['inc_revenue'])

# Debt Issurance to Assets Q
df['debt_issuance_to_assets_q'] = safe_div_positive(df['cf_issuance_of_debt'], df['bal_total_assets'])

# Debt Repayment to Assets Q
df['debt_repayment_to_assets_q'] = safe_div_positive(df['cf_repayment_of_debt'], df['bal_total_assets'])

# Net Debt Finance to Assets Q
df['net_debt_financing_to_assets_q'] = safe_div_positive(
    (df['cf_issuance_of_debt'] - df['cf_repayment_of_debt']),
    df['bal_total_assets']
)

# Finance Cash Flows to Assets Q
df['financing_cash_flows_to_assets_q'] = safe_div_positive(df['cf_net_cash_from_financing'], df['bal_total_assets'])

# Dividend Payout Ratio Q
df['dividend_payout_ratio_q'] = safe_div_positive(
    (df['inc_dividends_per_share'] * df['shr_weighted_average_basic_shares']),                 # I DONT LIKE THIS, CHANGE LATER => Solved
    df['inc_net_income_available_to_common']
)

# Dividend Yield Q
df['dividend_yield_q'] = safe_div_positive(df['inc_dividends_per_share'], df['price_close_q'])              # I DONT LIKE THIS, CHANGE LATER => Solved

# Shareholder Yield Q
df['shareholder_yield_q'] = (df['dividend_yield_q'] + df['buyback_to_market_cap_q'])       # I DONT LIKE THIS, CHANGE LATER => Solved

# Share Count Change Percentage Q
df['share_count_change_pct_q'] = df['shr_shares_change_percentage'] 

# Dilution Ratio Q
df['diluted_ratio_q'] = df['shr_dilution_percentage']

# Treasury Shares Ratio Q
df['treasury_shares_ratio_q'] = safe_div_positive(df['shr_treasury_shares'], df['bal_shares_issued'])


###################################################
# 13) Per Share Features
###################################################

# Sales per Share Q
df['sales_per_share_q'] = safe_div_positive(df['inc_revenue'], df['shr_weighted_average_basic_shares'])

# OCF per Share Q
df['OCF_per_share_q'] = safe_div_positive(df['cf_net_cash_from_operations'], df['shr_weighted_average_basic_shares'])

# FCF per Share Q
df['FCF_per_share_q'] = safe_div_positive(df['cf_free_cash_flow'], df['shr_weighted_average_basic_shares'])

# Tangible Book Value per Share Q
df['tangible_book_value_per_share_q'] = safe_div_positive(df['bal_tangible_book_value'], df['shr_basic_shares_outstanding']) # I DONT LIKE THIS, CHANGE LATER

# Cash per Share Q
df['cash_per_share_q'] = safe_div_positive(df['bal_cash_and_equivalents'], df['shr_basic_shares_outstanding']) # I DONT LIKE THIS, CHANGE LATER

# Net Debt per Share Q
df['net_debt_per_share_q'] = safe_div_positive(df['bal_net_debt'], df['shr_basic_shares_outstanding']) # I DONT LIKE THIS, CHANGE LATER


###################################################
# 14) Valuation Multiples
###################################################

# P/E Q
# Only keep when diluted EPS is positive so the multiple stays economically meaningful
df['P/E_q'] = safe_div_positive(df['price_close_q'], df['inc_diluted_eps'])

# P/B Q
# Only keep when book value per share is positive so the multiple stays economically meaningful
df['P/B_q'] = safe_div_positive(df['price_close_q'], df['bal_book_value_per_share']) # I DONT LIKE THIS, CHANGE LATER

#P/TBV Q
# Only keep when tangible book value is positive so the multiple stays economically meaningful
df['P/TBV_q'] = safe_div_positive(df['basic_market_cap_q'], df['bal_tangible_book_value']) # I DONT LIKE THIS, CHANGE LATER

# P/S Q
df['P/S_q'] = safe_div_positive(df['basic_market_cap_q'], df['inc_revenue']) # I DONT LIKE THIS, CHANGE LATER

# P/OCF_q
df['P/OCF_q'] = safe_div_positive(df['basic_market_cap_q'], df['cf_net_cash_from_operations']) # I DONT LIKE THIS, CHANGE LATER

# P/FCF_q
df['P/FCF_q'] = safe_div_positive(df['basic_market_cap_q'], df['cf_free_cash_flow']) # I DONT LIKE THIS, CHANGE LATER

# Earnings Yield Q
df['earnings_yield_q'] = safe_div_positive(df['inc_net_income_available_to_common'], df['diluted_market_cap_q'])    # I DONT LIKE THIS, CHANGE LATER

# OCF Yield Q
df['OCF_yield_q'] = safe_div_positive(df['cf_net_cash_from_operations'], df['basic_market_cap_q']) # I DONT LIKE THIS, CHANGE LATER

# FCF Yield Q
df['FCF_yield_q'] = safe_div_positive(df['cf_free_cash_flow'], df['basic_market_cap_q']) # I DONT LIKE THIS, CHANGE LATER

# EV/Sales Q
df['EV/Sales_q'] = safe_div_positive(df['enterprise_value_q'], df['inc_revenue'])

# EV/EBIT Q
df['EV/EBIT_q'] = safe_div_positive(df['enterprise_value_q'], df['inc_operating_income'])

# EV/EBITDA Q
df['EV/EBITDA_q'] = safe_div_positive(
    df['enterprise_value_q'],
    (df['inc_operating_income'] + df['inc_depreciation_amortization'])
)


###################################################
# 15) Growth Features
###################################################

# Revenue Growth QoQ
df['revenue_growth_qoq'] = safe_div_positive(
    df['inc_revenue'],
    df.groupby('ticker')['inc_revenue'].shift(1)
) - 1

# Gross Profit Growth QoQ
df['gross_profit_growth_qoq'] = safe_div_positive(
    df['inc_gross_profit'],
    df.groupby('ticker')['inc_gross_profit'].shift(1)
) - 1

# Operating Income Growth QoQ
df['operating_income_growth_qoq'] = safe_div_positive(
    df['inc_operating_income'],
    df.groupby('ticker')['inc_operating_income'].shift(1)
) - 1

# Net Income Growth QoQ
df['net_income_growth_qoq'] = safe_div_positive(
    df['inc_net_income'],
    df.groupby('ticker')['inc_net_income'].shift(1)
) - 1

# OCF Growth QoQ
df['OCF_growth_qoq'] = safe_div_positive(
    df['cf_net_cash_from_operations'],
    df.groupby('ticker')['cf_net_cash_from_operations'].shift(1)
) - 1

# FCF Growth QoQ
df['FCF_growth_qoq'] = safe_div_positive(
    df['cf_free_cash_flow'],
    df.groupby('ticker')['cf_free_cash_flow'].shift(1)
) - 1

# EPS Growth QoQ
df['EPS_growth_qoq'] = safe_div_positive(
    df['inc_diluted_eps'],
    df.groupby('ticker')['inc_diluted_eps'].shift(1)
) - 1

# Assets Growth QoQ
df['assets_growth_qoq'] = safe_div_positive(
    df['bal_total_assets'],
    df.groupby('ticker')['bal_total_assets'].shift(1)
) - 1

# Debt Growth QoQ
df['debt_growth_qoq'] = safe_div_positive(
    df['bal_total_debt'],
    df.groupby('ticker')['bal_total_debt'].shift(1)
) - 1

# Revenue Growth YoY
df['revenue_growth_yoy'] = safe_div_positive(
    df['inc_revenue'],
    df.groupby('ticker')['inc_revenue'].shift(4)
) - 1

# Gross Profit Growth YoY
df['gross_profit_growth_yoy'] = safe_div_positive(
    df['inc_gross_profit'],
    df.groupby('ticker')['inc_gross_profit'].shift(4)
) - 1

# Operating Income Growth YoY
df['operating_income_growth_yoy'] = safe_div_positive(
    df['inc_operating_income'],
    df.groupby('ticker')['inc_operating_income'].shift(4)
) - 1

# Net Income Growth YoY
df['net_income_growth_yoy'] = safe_div_positive(
    df['inc_net_income'],
    df.groupby('ticker')['inc_net_income'].shift(4)
) - 1

# OCF Growth YoY
df['OCF_growth_yoy'] = safe_div_positive(
    df['cf_net_cash_from_operations'],
    df.groupby('ticker')['cf_net_cash_from_operations'].shift(4)
) - 1

# FCF Growth YoY
df['FCF_growth_yoy'] = safe_div_positive(
    df['cf_free_cash_flow'],
    df.groupby('ticker')['cf_free_cash_flow'].shift(4)
) - 1

# EPS Growth YoY
df['EPS_growth_yoy'] = safe_div_positive(
    df['inc_diluted_eps'],
    df.groupby('ticker')['inc_diluted_eps'].shift(4)
) - 1

# Assets Growth YoY
df['assets_growth_yoy'] = safe_div_positive(
    df['bal_total_assets'],
    df.groupby('ticker')['bal_total_assets'].shift(4)
) - 1

# Debt Growth YoY
df['debt_growth_yoy'] = safe_div_positive(
    df['bal_total_debt'],
    df.groupby('ticker')['bal_total_debt'].shift(4)
) - 1

# Share Percentage Change YoY
df['shr_pct_change_yoy'] = safe_div_positive(
    df['shr_basic_shares_outstanding'],
    df.groupby('ticker')['shr_basic_shares_outstanding'].shift(4)       # I DONT LIKE THIS, CHANGE LATER
) - 1


###################################################
# 16) Quality and Accounting Risk Proxies
###################################################

# Accural Ratio Q
df['accrual_ratio_q'] = safe_div_positive(
    (df['inc_net_income'] - df['cf_net_cash_from_operations']),
    df['bal_total_assets']
)

# Stock Compensation to Revenue Q
df['stock_compensation_to_revenue_q'] = safe_div_positive(df['inc_stock_based_compensation'], df['inc_revenue'])

# Goodwill to Assets Q
df['goodwill_to_assets_q'] = safe_div_positive(df['bal_goodwill'], df['bal_total_assets'])

# Intangibles to Assets Q
df['intangibles_to_assets_q'] = safe_div_positive(df['bal_intangible_assets'], df['bal_total_assets'])

# Accumulated Other Comprehensive Income to Equity Q
df['AOCI_to_equity_q'] = safe_div_positive(df['bal_accumulated_other_comprehensive_income'], df['bal_total_equity'])

# Impairments to Assets Q
df['impairments_to_assets_q'] = safe_div_positive(df['inc_impairment_charges'], df['bal_total_assets'])

# Deferred Revenye to Revenue Q
df['deferred_revenue_to_revenue_q'] = safe_div_positive(
    (df['bal_deferred_revenue_current'] + df['bal_deferred_revenue_non_current']),
    df['inc_revenue']
)

# Prepaid Expenses to Assets Q
df['prepaid_expenses_to_assets_q'] = safe_div_positive(df['bal_prepaid_expenses'], df['bal_total_assets'])

df = df.copy() # Just to be safe with the SettingWithCopyWarning appearing from pandas


In [9]:
# We will now add sector specific ratios to the dataset

# Note: After creating a list of all possible sector-industry ratios/multiples from the previous column list, I then created a non-code formula
#       for each sector ratio/multiple. I then coded the first sector ratio/multiples so that I could give to ChatGPT to code the rest of the 
#       sector ratios/formulas from the list I made. There's a lot of sector-industry ratios/multiples!




# Safe Division Function
def safe_div(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    result = numerator / denominator
    return result.replace([np.inf, -np.inf], np.nan)

# Safe Division Function for ratios that only make sense with a positive denominator
def safe_div_positive(numerator, denominator):
    denominator = denominator.where(denominator > 0, np.nan)
    return safe_div(numerator, denominator)


###################################################
# Industry Lists
###################################################
bank_industries = [
    'National Commercial Banks',
    'State Commercial Banks',
]
insurance_industries = [
    'Fire, Marine & Casualty Insurance',
    'Life Insurance',
    'Accident & Health Insurance',
    'Insurance Carriers, NEC',
    'Hospital & Medical Service Plans',
]

###################################################
# 1) Utilities - Core Block
###################################################

# Fuel Cost Ratio Q
df['fuel_cost_ratio_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_fuel_costs_generation'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Purchased Power Ratio Q
df['purchased_power_ratio_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_purchased_power_costs'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Transmission Expense Ratio Q
df['transmission_expense_ratio_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_transmission_expenses'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Distribution Expense Ratio Q
df['distribution_expense_ratio_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_distribution_expenses'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Storm Damage Ratio Q
df['storm_damage_ratio_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_storm_damage_costs'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Rate Base Turnover Q
df['rate_base_turnover_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_electric_utility_revenue'], df['bal_rate_base']),
    np.nan,
)

# Revenue per MWh Q
df['revenue_per_mwh_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_electric_utility_revenue'], df['inc_megawatt_hours_sold']),
    np.nan,
)

# Regulatory Assets to Assets Q
df['regulatory_assets_to_assets_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive((df['bal_regulatory_assets_current'] + df['bal_regulatory_assets_noncurrent']), df['bal_total_assets']),
    np.nan,
)

# Regulatory Liabilities to Liabilities Q
df['regulatory_liabilities_to_liabilities_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive((df['bal_regulatory_liabilities_current'] + df['bal_regulatory_liabilities_noncurrent']), df['bal_total_liabilities']),
    np.nan,
)

###################################################
# 1A) Utilities - Extras
###################################################

# Vegetation Management Ratio Q
df['vegetation_management_ratio_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_vegetation_management_costs'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Renewable Energy Credit Revenue Share Q
df['renewable_energy_credit_revenue_share_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_renewable_energy_credit_revenue'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Gas Utility Revenue Share Q
df['gas_utility_revenue_share_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_gas_utility_revenue'], df['inc_revenue']),
    np.nan,
)

# Allowed Rate of Return Q
df['allowed_rate_of_return_q'] = np.where(
    df['sector'] == 'Utilities',
    df['inc_allowed_rate_of_return'],
    np.nan,
)

# Renewable Energy Percentage Q
df['renewable_energy_percentage_q'] = np.where(
    df['sector'] == 'Utilities',
    df['inc_renewable_energy_percentage'],
    np.nan,
)

# Revenue per Customer Q
df['revenue_per_customer_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_electric_utility_revenue'], df['inc_customers_total']),
    np.nan,
)

# MWh per Customer Q
df['mwh_per_customer_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_megawatt_hours_sold'], df['inc_customers_total']),
    np.nan,
)

# Residential Revenue Mix Q
df['residential_revenue_mix_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_electric_revenue_residential'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Commercial Revenue Mix Q
df['commercial_revenue_mix_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_electric_revenue_commercial'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Industrial Revenue Mix Q
df['industrial_revenue_mix_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_electric_revenue_industrial'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Wholesale Revenue Mix Q
df['wholesale_revenue_mix_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_electric_revenue_wholesale'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Generation Plant Intensity Q
df['generation_plant_intensity_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['bal_electric_generation_plant_gross'], df['bal_total_assets']),
    np.nan,
)

# Transmission Plant Intensity Q
df['transmission_plant_intensity_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['bal_electric_transmission_plant_gross'], df['bal_total_assets']),
    np.nan,
)

# Distribution Plant Intensity Q
df['distribution_plant_intensity_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['bal_electric_distribution_plant_gross'], df['bal_total_assets']),
    np.nan,
)

# CWIP to Assets Q
df['cwip_to_assets_utility_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['bal_construction_work_in_progress'], df['bal_total_assets']),
    np.nan,
)

# Deferred Fuel Assets to Assets Q
df['deferred_fuel_assets_to_assets_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['bal_deferred_fuel_costs_asset'], df['bal_total_assets']),
    np.nan,
)

# Deferred Fuel Liabilities to Liabilities Q
df['deferred_fuel_liabilities_to_liabilities_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['bal_deferred_fuel_costs_liability'], df['bal_total_liabilities']),
    np.nan,
)

# Generation Capacity Turnover Q
df['generation_capacity_turnover_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['inc_electric_utility_revenue'], df['bal_generation_capacity_megawatts']),
    np.nan,
)

# Nuclear Decommissioning Trust to Assets Q
df['nuclear_decommissioning_trust_to_assets_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['bal_nuclear_decommissioning_trust'], df['bal_total_assets']),
    np.nan,
)

# Pension Obligations to Assets Q
df['pension_obligations_to_assets_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['bal_pension_benefit_obligations'], df['bal_total_assets']),
    np.nan,
)

# Regulatory Asset Deferrals to Revenue Q
df['regulatory_asset_deferrals_to_revenue_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['cf_regulatory_asset_deferrals'], df['inc_electric_utility_revenue']),
    np.nan,
)

# Regulatory Asset Amortization to Revenue Q
df['regulatory_asset_amortization_to_revenue_q'] = np.where(
    df['sector'] == 'Utilities',
    safe_div_positive(df['cf_regulatory_asset_amortization'], df['inc_electric_utility_revenue']),
    np.nan,
)

###################################################
# 2) Banks - Core Block
###################################################

# Net Interest Margin Proxy Q
df['net_interest_margin_proxy_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_net_interest_income'], df['bal_average_interest_earning_assets']),
    np.nan,
)

# Provision Ratio Q
df['provision_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_provision_for_credit_losses'], df['inc_net_interest_income']),
    np.nan,
)

# Fee Income Share Q
df['fee_income_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_fee_income'], (df['inc_net_interest_income'] + df['inc_fee_income'] + df['inc_trading_revenue'])),
    np.nan,
)

# Trading Revenue Share Q
df['trading_revenue_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_trading_revenue'], df['inc_noninterest_income_total']),
    np.nan,
)

# Noninterest Income Share Q
df['noninterest_income_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_noninterest_income_total'], (df['inc_net_interest_income'] + df['inc_noninterest_income_total'])),
    np.nan,
)

# Efficiency Ratio Q
df['efficiency_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_noninterest_expense_total'], (df['inc_net_interest_income'] + df['inc_noninterest_income_total'])),
    np.nan,
)

# Compensation Ratio Q
df['compensation_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_compensation_and_benefits'], df['inc_noninterest_income_total']),
    np.nan,
)

# Loan to Deposit Q
df['loan_to_deposit_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_loans_held_for_investment'], df['bal_deposits_total']),
    np.nan,
)

# Allowance to Loans Q
df['allowance_to_loans_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_allowance_for_credit_losses'], df['bal_loans_held_for_investment']),
    np.nan,
)

# NPL Ratio Q
df['npl_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_nonperforming_loans'], df['bal_loans_held_for_investment']),
    np.nan,
)

# Net Charge-Off Ratio Q
df['net_charge_off_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_net_charge_offs'], df['bal_average_loans']),
    np.nan,
)

# Securities to Assets Q
df['securities_to_assets_bank_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive((df['bal_debt_securities_available_for_sale'] + df['bal_debt_securities_held_to_maturity']), df['bal_total_assets']),
    np.nan,
)

# Deposit Funding Mix Q
df['deposit_funding_mix_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_noninterest_bearing_deposits'], df['bal_deposits_total']),
    np.nan,
)

# Interest Bearing Deposit Share Q
df['interest_bearing_deposit_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_interest_bearing_deposits'], df['bal_deposits_total']),
    np.nan,
)

###################################################
# 2A) Banks - Extras
###################################################

# Reported Net Interest Margin Q
df['reported_net_interest_margin_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    df['inc_net_interest_margin'],
    np.nan,
)

# Reported ROAA Q
df['reported_roaa_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    df['inc_return_on_average_assets'],
    np.nan,
)

# Reported ROAE Q
df['reported_roae_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    df['inc_return_on_average_equity'],
    np.nan,
)

# Commercial Loan Income Mix Q
df['commercial_loan_income_mix_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_interest_income_loans_commercial'], df['inc_interest_income_loans_total']),
    np.nan,
)

# Consumer Loan Income Mix Q
df['consumer_loan_income_mix_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_interest_income_loans_consumer'], df['inc_interest_income_loans_total']),
    np.nan,
)

# Securities Income Share Q
df['securities_income_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_interest_income_securities'], (df['inc_interest_income_loans_total'] + df['inc_interest_income_securities'])),
    np.nan,
)

# Deposit Interest Burden Q
df['deposit_interest_burden_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_interest_expense_deposits'], df['bal_average_deposits']),
    np.nan,
)

# Borrowing Interest Burden Q
df['borrowing_interest_burden_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive((df['inc_interest_expense_borrowings'] + df['inc_interest_expense_long_term_debt']), df['bal_total_debt']),
    np.nan,
)

# Investment Banking Revenue Share Q
df['investment_banking_revenue_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_investment_banking_revenue'], df['inc_noninterest_income_total']),
    np.nan,
)

# Asset Management Fee Share Q
df['asset_management_fee_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_asset_management_fees'], df['inc_noninterest_income_total']),
    np.nan,
)

# Service Charges Share Q
df['service_charges_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_service_charges_on_deposits'], df['inc_noninterest_income_total']),
    np.nan,
)

# Card Income Share Q
df['card_income_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_card_income'], df['inc_noninterest_income_total']),
    np.nan,
)

# Mortgage Servicing Fee Share Q
df['mortgage_servicing_fee_share_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_mortgage_servicing_fees'], df['inc_noninterest_income_total']),
    np.nan,
)

# Occupancy Expense Ratio Q
df['occupancy_expense_ratio_bank_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_occupancy_expense'], df['inc_noninterest_income_total']),
    np.nan,
)

# Technology Expense Ratio Q
df['technology_expense_ratio_bank_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_technology_expense'], df['inc_noninterest_income_total']),
    np.nan,
)

# Professional Fees Ratio Q
df['professional_fees_ratio_bank_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_professional_fees'], df['inc_noninterest_income_total']),
    np.nan,
)

# Marketing Expense Ratio Q
df['marketing_expense_ratio_bank_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_marketing_expense'], df['inc_noninterest_income_total']),
    np.nan,
)

# Deposit Insurance Expense Ratio Q
df['deposit_insurance_expense_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['inc_deposit_insurance_expense'], df['inc_noninterest_income_total']),
    np.nan,
)

# Financing Receivables to Assets Q
df['financing_receivables_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_financing_receivables_before_allowance'], df['bal_total_assets']),
    np.nan,
)

# Trading Assets to Assets Q
df['trading_assets_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_trading_assets'], df['bal_total_assets']),
    np.nan,
)

# Trading Securities to Assets Q
df['trading_securities_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_trading_assets_securities'], df['bal_total_assets']),
    np.nan,
)

# Cash and Due from Banks to Assets Q
df['cash_due_from_banks_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_cash_and_due_from_banks'], df['bal_total_assets']),
    np.nan,
)

# Interest Bearing Deposits with Banks to Assets Q
df['interest_bearing_deposits_with_banks_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_interest_bearing_deposits_with_banks'], df['bal_total_assets']),
    np.nan,
)

# Fed Funds Sold and Resale to Assets Q
df['fedfunds_sold_resale_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_federal_funds_sold_and_securities_resale'], df['bal_total_assets']),
    np.nan,
)

# Loans Net to Assets Q
df['loans_net_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_loans_net'], df['bal_total_assets']),
    np.nan,
)

# Nonperforming Assets Ratio Q
df['nonperforming_assets_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_nonperforming_assets'], df['bal_total_assets']),
    np.nan,
)

# CET1 Ratio Q
df['cet1_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    df['bal_common_equity_tier_1_ratio'],
    np.nan,
)

# Tier 1 Capital Ratio Q
df['tier1_capital_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    df['bal_tier_1_capital_ratio'],
    np.nan,
)

# Risk Weighted Assets to Assets Q
df['risk_weighted_assets_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_total_risk_weighted_assets'], df['bal_total_assets']),
    np.nan,
)

# Bank Leverage Ratio Q
df['bank_leverage_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    df['bal_leverage_ratio'],
    np.nan,
)

# Deposits to Liabilities Q
df['deposits_to_liabilities_bank_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['bal_deposits_total'], df['bal_total_liabilities']),
    np.nan,
)

# Wholesale Funding Dependence Q
df['wholesale_funding_dependence_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive((df['bal_federal_funds_purchased_and_repurchase_agreements'] + df['bal_short_term_borrowings'] + df['bal_subordinated_debt']), df['bal_total_liabilities']),
    np.nan,
)

# Loan Growth Bank Q
df['loan_growth_bank_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['cf_net_increase_in_loans'], df['bal_loans_held_for_investment']),
    np.nan,
)

# Deposit Growth Bank Q
df['deposit_growth_bank_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['cf_net_increase_in_deposits'], df['bal_deposits_total']),
    np.nan,
)

# Securities Purchases to Assets Q
df['securities_purchases_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['cf_purchases_of_securities'], df['bal_total_assets']),
    np.nan,
)

# Securities Sales to Assets Q
df['securities_sales_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['cf_proceeds_from_sales_of_securities'], df['bal_total_assets']),
    np.nan,
)

# Securities Maturities to Assets Q
df['securities_maturities_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(bank_industries)),
    safe_div_positive(df['cf_proceeds_from_maturities_of_securities'], df['bal_total_assets']),
    np.nan,
)

###################################################
# 3) Insurance - Core Block
###################################################

# Premium Retention Ratio Q
df['premium_retention_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_premiums_written_net'], df['inc_premiums_written_gross']),
    np.nan,
)

# Ceded Premium Ratio Q
df['ceded_premium_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_premiums_ceded_to_reinsurers'], df['inc_premiums_written_gross']),
    np.nan,
)

# Underwriting Margin Q
df['underwriting_margin_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_underwriting_income'], df['inc_premiums_earned']),
    np.nan,
)

# Claims Ratio Q
df['claims_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_policy_benefits_claims_incurred'], df['inc_premiums_earned']),
    np.nan,
)

# Investment Income Ratio Q
df['investment_income_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(( df['inc_net_investment_income_fixed_maturity'] + df['inc_net_investment_income_equity_securities'] + df['inc_net_investment_income_mortgage_loans'] + df['inc_net_investment_income_policy_loans'] ), df['bal_total_invested_assets']),
    np.nan,
)

# Reserves to Premiums Q
df['reserves_to_premiums_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive((df['bal_claim_reserves_unpaid_losses'] + df['bal_claim_reserves_loss_adjustment_expenses']), df['inc_premiums_earned']),
    np.nan,
)

# DAC to Premiums Q
df['dac_to_premiums_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_deferred_acquisition_costs_asset'], df['inc_premiums_earned']),
    np.nan,
)

# Reinsurance Recoverables to Reserves Q
df['reinsurance_recoverables_to_reserves_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_reinsurance_recoverables'], (df['bal_claim_reserves_unpaid_losses'] + df['bal_claim_reserves_loss_adjustment_expenses'])),
    np.nan,
)

# Policy Loans to Assets Q
df['policy_loans_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_policy_loans'], df['bal_total_assets']),
    np.nan,
)

# Statutory Surplus to Assets Q
df['statutory_surplus_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_statutory_surplus'], df['bal_total_assets']),
    np.nan,
)

###################################################
# 3A) Insurance - Extras
###################################################

# Life Premium Mix Q
df['life_premium_mix_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_premium_revenue_life_insurance'], df['inc_premiums_earned']),
    np.nan,
)

# Annuity Premium Mix Q
df['annuity_premium_mix_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_premium_revenue_annuities'], df['inc_premiums_earned']),
    np.nan,
)

# Property Casualty Premium Mix Q
df['property_casualty_premium_mix_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_premium_revenue_property_casualty'], df['inc_premiums_earned']),
    np.nan,
)

# Health Premium Mix Q
df['health_premium_mix_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_premium_revenue_health'], df['inc_premiums_earned']),
    np.nan,
)

# Death Benefits Ratio Q
df['death_benefits_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_death_benefits_paid'], df['inc_premiums_earned']),
    np.nan,
)

# Annuity Benefits Ratio Q
df['annuity_benefits_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_annuity_benefits_paid'], df['inc_premiums_earned']),
    np.nan,
)

# Property Casualty Losses Ratio Q
df['property_casualty_losses_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_claims_losses_property_casualty'], df['inc_premiums_earned']),
    np.nan,
)

# Policy Interest Credited Ratio Q
df['policy_interest_credited_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_policy_interest_credited'], df['inc_premiums_earned']),
    np.nan,
)

# Acquisition Cost Deferral Ratio Q
df['acquisition_cost_deferral_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_acquisition_costs_deferred'], df['inc_premiums_earned']),
    np.nan,
)

# Acquisition Cost Amortization Ratio Q
df['acquisition_cost_amortization_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_acquisition_costs_amortization'], df['inc_premiums_earned']),
    np.nan,
)

# Underwriting Expense Ratio Q
df['underwriting_expense_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_underwriting_expenses'], df['inc_premiums_earned']),
    np.nan,
)

# Combined Ratio Proxy Q
df['combined_ratio_proxy_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive((df['inc_policy_benefits_claims_incurred'] + df['inc_underwriting_expenses']), df['inc_premiums_earned']),
    np.nan,
)

# Reported Combined Ratio Q
df['reported_combined_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    df['inc_combined_ratio_property_casualty'],
    np.nan,
)

# Reported Loss Ratio Q
df['reported_loss_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    df['inc_loss_ratio'],
    np.nan,
)

# Reported Expense Ratio Q
df['reported_expense_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    df['inc_expense_ratio'],
    np.nan,
)

# Reported Insurance ROE Q
df['reported_insurance_roe_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    df['inc_return_on_equity'],
    np.nan,
)

# Reported Investment Yield Q
df['reported_investment_yield_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    df['inc_investment_yield'],
    np.nan,
)

# New Business Value to Premiums Q
df['new_business_value_to_premiums_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_new_business_value'], df['inc_premiums_earned']),
    np.nan,
)

# Policy Surrenders to Balances Q
df['policy_surrenders_to_balances_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_policy_surrenders_withdrawals'], df['bal_policyholder_account_balances_annuities']),
    np.nan,
)

# Universal Life Deposits to Balances Q
df['universal_life_deposits_to_balances_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_universal_life_deposits'], df['bal_policyholder_account_balances_annuities']),
    np.nan,
)

# Premiums per Policy in Force Q
df['premiums_per_policy_in_force_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['inc_premiums_earned'], df['inc_policies_in_force_count']),
    np.nan,
)

# Future Policy Benefits to Assets Q
df['future_policy_benefits_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_future_policy_benefits_life'], df['bal_total_assets']),
    np.nan,
)

# Policyholder Account Balances to Assets Q
df['policyholder_account_balances_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_policyholder_account_balances_annuities'], df['bal_total_assets']),
    np.nan,
)

# Unearned Premiums to Written Q
df['unearned_premiums_to_written_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_unearned_premiums_liability'], df['inc_premiums_written_gross']),
    np.nan,
)

# Separate Account Liabilities to Liabilities Q
df['separate_account_liabilities_to_liabilities_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_separate_account_liabilities'], df['bal_total_liabilities']),
    np.nan,
)

# Separate Account Assets to Assets Q
df['separate_account_assets_to_assets_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_separate_account_assets'], df['bal_total_assets']),
    np.nan,
)

# Premiums Receivable to Written Q
df['premiums_receivable_to_written_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_premiums_receivable'], df['inc_premiums_written_gross']),
    np.nan,
)

# Risk Based Capital Ratio Q
df['risk_based_capital_ratio_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    df['bal_risk_based_capital_ratio'],
    np.nan,
)

# Dividends from Insurance Subsidiaries to Surplus Q
df['dividends_from_insurance_subsidiaries_to_surplus_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_dividends_from_insurance_subsidiaries'], df['bal_statutory_surplus']),
    np.nan,
)

# Reinsurance Premiums Assumed to Gross Written Q
df['reinsurance_premiums_assumed_to_gross_written_q'] = np.where(
    (df['sector'] == 'Finance') & (df['industry'].isin(insurance_industries)),
    safe_div_positive(df['bal_reinsurance_premiums_assumed'], df['inc_premiums_written_gross']),
    np.nan,
)

###################################################
# 4) Real Estate / REITs - Core Block
###################################################

# NOI Margin Q
df['noi_margin_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_net_operating_income'], df['inc_rental_income_properties']),
    np.nan,
)

# Property Opex Ratio Q
df['property_opex_ratio_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_property_operating_expenses'], df['inc_rental_income_properties']),
    np.nan,
)

# Real Estate Tax Ratio Q
df['real_estate_tax_ratio_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_real_estate_taxes'], df['inc_rental_income_properties']),
    np.nan,
)

# FFO Margin Q
df['ffo_margin_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_funds_from_operations'], df['inc_rental_income_properties']),
    np.nan,
)

# AFFO Margin Q
df['affo_margin_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_adjusted_funds_from_operations'], df['inc_rental_income_properties']),
    np.nan,
)

# FFO Yield Q
df['ffo_yield_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_funds_from_operations'], df['diluted_market_cap_q']),
    np.nan,
)

# AFFO Yield Q
df['affo_yield_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_adjusted_funds_from_operations'], df['diluted_market_cap_q']),
    np.nan,
)

# P/FFO Q
df['p_ffo_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['diluted_market_cap_q'], df['inc_funds_from_operations']),
    np.nan,
)

# P/AFFO Q
df['p_affo_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['diluted_market_cap_q'], df['inc_adjusted_funds_from_operations']),
    np.nan,
)

# Investment Property Leverage Q
df['investment_property_leverage_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_total_debt'], df['bal_investment_properties_net']),
    np.nan,
)

# Debt to Fair Value Q
df['debt_to_fair_value_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_total_debt'], df['bal_investment_properties_fair_value']),
    np.nan,
)

# Straight Line Rent Receivable Ratio Q
df['straight_line_rent_receivable_ratio_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_straight_line_rent_receivable'], df['inc_rental_income_properties']),
    np.nan,
)

# Development Cost Ratio Q
df['development_cost_ratio_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_properties_under_development_cost'], df['bal_total_assets']),
    np.nan,
)

###################################################
# 4A) Real Estate / REITs - Extras
###################################################

# Same Store NOI Margin Q
df['same_store_noi_margin_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_same_store_net_operating_income'], df['inc_rental_income_properties']),
    np.nan,
)

# FAD Margin Q
df['fad_margin_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_funds_available_for_distribution'], df['inc_rental_income_properties']),
    np.nan,
)

# FAD Yield Q
df['fad_yield_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_funds_available_for_distribution'], df['diluted_market_cap_q']),
    np.nan,
)

# P/FAD Q
df['p_fad_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['diluted_market_cap_q'], df['inc_funds_available_for_distribution']),
    np.nan,
)

# Reported FFO per Share Q
df['ffo_per_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_ffo_per_share'],
    np.nan,
)

# Reported AFFO per Share Q
df['affo_per_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_affo_per_share'],
    np.nan,
)

# Cap Rate Q
df['cap_rate_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_cap_rate'],
    np.nan,
)

# Base Rent Share Q
df['base_rent_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_rental_income_base_rent'], df['inc_rental_income_properties']),
    np.nan,
)

# Percentage Rent Share Q
df['percentage_rent_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_rental_income_percentage_rent'], df['inc_rental_income_properties']),
    np.nan,
)

# Tenant Reimbursements Share Q
df['tenant_reimbursements_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_rental_income_tenant_reimbursements'], df['inc_rental_income_properties']),
    np.nan,
)

# Straight Line Adjustment Share Q
df['straight_line_adjustment_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_rental_income_straight_line_adjustment'], df['inc_rental_income_properties']),
    np.nan,
)

# Portfolio Occupancy Q
df['portfolio_occupancy_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_occupancy_rate_portfolio'],
    np.nan,
)

# Same Store Occupancy Q
df['same_store_occupancy_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_occupancy_rate_same_store'],
    np.nan,
)

# Leased Rate Portfolio Q
df['leased_rate_portfolio_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_leased_rate_portfolio'],
    np.nan,
)

# Same Store NOI Growth Q
df['same_store_noi_growth_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_same_store_noi_growth'],
    np.nan,
)

# Lease Expirations Year 1 Q
df['lease_expirations_year_1_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_lease_expirations_year_1'],
    np.nan,
)

# Average Remaining Lease Term Years Q
df['average_remaining_lease_term_years_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_average_remaining_lease_term_years'],
    np.nan,
)

# Top 10 Tenants Percentage of Rent Q
df['top_10_tenants_percentage_of_rent_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_top_10_tenants_percentage_of_rent'],
    np.nan,
)

# Largest Tenant Percentage Q
df['largest_tenant_percentage_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_largest_tenant_percentage'],
    np.nan,
)

# Weighted Average Lease Term by Rent Q
df['weighted_average_lease_term_by_rent_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_weighted_average_lease_term_by_rent'],
    np.nan,
)

# Average Annual Rent Escalation Percentage Q
df['average_annual_rent_escalation_percentage_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_average_annual_rent_escalation_percentage'],
    np.nan,
)

# Tenant Retention Rate Q
df['tenant_retention_rate_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['inc_tenant_retention_rate'],
    np.nan,
)

# Real Estate at Cost to Assets Q
df['real_estate_at_cost_to_assets_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_real_estate_investments_at_cost'], df['bal_total_assets']),
    np.nan,
)

# Accumulated Depreciation Real Estate to Cost Q
df['accumulated_depreciation_real_estate_to_cost_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_accumulated_depreciation_real_estate'], df['bal_real_estate_investments_at_cost']),
    np.nan,
)

# Held for Sale to Assets Q
df['held_for_sale_to_assets_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_real_estate_held_for_sale'], df['bal_total_assets']),
    np.nan,
)

# Investment Properties Net to Assets Q
df['investment_properties_net_to_assets_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_investment_properties_net'], df['bal_total_assets']),
    np.nan,
)

# In Place Lease Intangibles to Assets Q
df['in_place_lease_intangibles_to_assets_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_intangible_assets_in_place_leases'], df['bal_total_assets']),
    np.nan,
)

# Above Market Leases to Assets Q
df['above_market_leases_to_assets_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_intangible_assets_above_market_leases'], df['bal_total_assets']),
    np.nan,
)

# Below Market Leases to Liabilities Q
df['below_market_leases_to_liabilities_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_intangible_liabilities_below_market_leases'], df['bal_total_liabilities']),
    np.nan,
)

# Accumulated Amortization Lease Intangibles Ratio Q
df['accumulated_amortization_lease_intangibles_ratio_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_accumulated_amortization_lease_intangibles'], (df['bal_intangible_assets_in_place_leases'] + df['bal_intangible_assets_above_market_leases'])),
    np.nan,
)

# Retail Property Mix Q
df['retail_property_mix_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_investment_properties_retail'], df['bal_investment_properties_net']),
    np.nan,
)

# Office Property Mix Q
df['office_property_mix_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_investment_properties_office'], df['bal_investment_properties_net']),
    np.nan,
)

# Industrial Property Mix Q
df['industrial_property_mix_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_investment_properties_industrial'], df['bal_investment_properties_net']),
    np.nan,
)

# Residential Property Mix Q
df['residential_property_mix_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_investment_properties_residential'], df['bal_investment_properties_net']),
    np.nan,
)

# Hotel Property Mix Q
df['hotel_property_mix_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_investment_properties_hotel'], df['bal_investment_properties_net']),
    np.nan,
)

# Revenue per Square Foot Q
df['revenue_per_square_foot_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_rental_income_properties'], df['bal_gross_leasable_area_square_feet']),
    np.nan,
)

# Revenue per Property Q
df['revenue_per_property_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_rental_income_properties'], df['bal_number_of_properties_owned']),
    np.nan,
)

# NOI per Property Q
df['noi_per_property_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_net_operating_income'], df['bal_number_of_properties_owned']),
    np.nan,
)

# Weighted Average Cost of Debt Q
df['weighted_average_cost_of_debt_q'] = np.where(
    df['sector'] == 'Real Estate',
    df['bal_weighted_average_cost_of_debt'],
    np.nan,
)

# Debt Maturity Within 1 Year Share Q
df['debt_maturity_within_1_year_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_debt_maturities_within_1_year'], df['bal_total_debt']),
    np.nan,
)

# Debt Maturity 1 to 3 Years Share Q
df['debt_maturity_1_to_3_years_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_debt_maturities_1_to_3_years'], df['bal_total_debt']),
    np.nan,
)

# Debt Maturity 3 to 5 Years Share Q
df['debt_maturity_3_to_5_years_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_debt_maturities_3_to_5_years'], df['bal_total_debt']),
    np.nan,
)

# Debt Maturity After 5 Years Share Q
df['debt_maturity_after_5_years_share_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['bal_debt_maturities_after_5_years'], df['bal_total_debt']),
    np.nan,
)

# Properties Under Development Count Intensity Q
df['properties_under_development_count_intensity_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_properties_under_development_count'], df['bal_number_of_properties_owned']),
    np.nan,
)

# Properties Acquired Count Intensity Q
df['properties_acquired_count_intensity_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_properties_acquired_during_period'], df['bal_number_of_properties_owned']),
    np.nan,
)

# Properties Disposed Count Intensity Q
df['properties_disposed_count_intensity_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['inc_properties_disposed_during_period'], df['bal_number_of_properties_owned']),
    np.nan,
)

# Redevelopment Capex Intensity Q
df['redevelopment_capex_intensity_q'] = np.where(
    df['sector'] == 'Real Estate',
    safe_div_positive(df['cf_redevelopment_capital_expenditures'], df['inc_rental_income_properties']),
    np.nan,
)

###################################################
# 5) Energy / Oil & Gas - Core Block
###################################################

# Upstream Revenue Share Q
df['upstream_revenue_share_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_revenue_upstream_segment'], df['inc_revenue']),
    np.nan,
)

# Downstream Revenue Share Q
df['downstream_revenue_share_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_revenue_downstream_segment'], df['inc_revenue']),
    np.nan,
)

# Midstream Revenue Share Q
df['midstream_revenue_share_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_revenue_midstream_segment'], df['inc_revenue']),
    np.nan,
)

# Chemicals Revenue Share Q
df['chemicals_revenue_share_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_revenue_chemicals_segment'], df['inc_revenue']),
    np.nan,
)

# Oil Mix Q
df['oil_mix_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_revenue_crude_oil_condensate'], df['inc_revenue']),
    np.nan,
)

# Gas Mix Q
df['gas_mix_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_revenue_natural_gas'], df['inc_revenue']),
    np.nan,
)

# Exploration Intensity Q
df['exploration_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_exploration_costs_expensed'], df['inc_revenue']),
    np.nan,
)

# Lifting Cost Burden Q
df['lifting_cost_burden_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_lifting_costs_per_boe'], df['inc_average_realized_price_oil_per_barrel']),
    np.nan,
)

# Refining Margin Spread Q
df['refining_margin_spread_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_refining_margin_per_barrel'], df['inc_average_realized_price_oil_per_barrel']),
    np.nan,
)

# PV10 to EV Q
df['pv10_to_ev_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_pv10_value_proved_reserves'], df['enterprise_value_q']),
    np.nan,
)

# Reserve Replacement Strength Q
df['reserve_replacement_strength_q'] = np.where(
    df['sector'] == 'Energy',
    df['inc_reserve_replacement_ratio'],
    np.nan,
)

# Asset Retirement Burden Q
df['asset_retirement_burden_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive((df['bal_asset_retirement_obligations_current'] + df['bal_asset_retirement_obligations_noncurrent']), df['bal_total_assets']),
    np.nan,
)

###################################################

# 5A) Energy / Oil & Gas - Extras

###################################################

# NGL Mix Q
df['ngl_mix_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_revenue_natural_gas_liquids'], df['inc_revenue']),
    np.nan,
)

# Refined Products Mix Q
df['refined_products_mix_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_revenue_refined_products'], df['inc_revenue']),
    np.nan,
)

# Exploration Development Intensity Q
df['exploration_development_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_exploration_development_costs'], df['inc_revenue']),
    np.nan,
)

# Upstream Production Cost Burden Q
df['upstream_production_cost_burden_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_production_costs_upstream'], df['inc_revenue_upstream_segment']),
    np.nan,
)

# Crude Purchase Burden Q
df['crude_purchase_burden_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_refining_costs_crude_oil_purchased'], df['inc_revenue_downstream_segment']),
    np.nan,
)

# Upstream DDA Burden Q
df['upstream_dda_burden_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_depletion_depreciation_amortization_upstream'], df['inc_revenue_upstream_segment']),
    np.nan,
)

# Oil Gas Impairment Burden Q
df['oil_gas_impairment_burden_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_impairment_charges_oil_gas_properties'], df['inc_revenue']),
    np.nan,
)

# Oil Production Intensity Q
df['oil_production_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_oil_production_volumes_barrels_per_day'], df['bal_total_assets']),
    np.nan,
)

# Gas Production Intensity Q
df['gas_production_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_natural_gas_production_volumes_mcf_per_day'], df['bal_total_assets']),
    np.nan,
)

# BOE Production Intensity Q
df['boe_production_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_oil_equivalent_production_boe_per_day'], df['bal_total_assets']),
    np.nan,
)

# Realized Oil Price Q
df['realized_oil_price_q'] = np.where(
    df['sector'] == 'Energy',
    df['inc_average_realized_price_oil_per_barrel'],
    np.nan,
)

# Realized Gas Price Q
df['realized_gas_price_q'] = np.where(
    df['sector'] == 'Energy',
    df['inc_average_realized_price_gas_per_mcf'],
    np.nan,
)

# Refinery Throughput Intensity Q
df['refinery_throughput_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_refinery_throughput_barrels_per_day'], df['bal_total_assets']),
    np.nan,
)

# Proved Oil Reserves Intensity Q
df['proved_oil_reserves_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_proved_reserves_oil_barrels'], df['bal_total_assets']),
    np.nan,
)

# Proved Gas Reserves Intensity Q
df['proved_gas_reserves_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_proved_reserves_natural_gas_mcf'], df['bal_total_assets']),
    np.nan,
)

# Reserve Life Q
df['reserve_life_q'] = np.where(
    df['sector'] == 'Energy',
    df['inc_reserve_life_years'],
    np.nan,
)

# EBITDAX Margin Q
df['ebitdax_margin_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_ebitdax'], df['inc_revenue']),
    np.nan,
)

# Finding Development Burden Q
df['finding_development_burden_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_finding_development_costs_per_boe'], df['inc_average_realized_price_oil_per_barrel']),
    np.nan,
)

# Commodity Derivative Impact Q
df['commodity_derivative_impact_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_commodity_derivative_gains_losses'], df['inc_revenue']),
    np.nan,
)

# Royalty Burden Q
df['royalty_burden_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['inc_royalty_expense'], df['inc_revenue']),
    np.nan,
)

# Proved Developed Properties to Assets Q
df['proved_developed_properties_to_assets_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['bal_oil_gas_properties_proved_developed'], df['bal_total_assets']),
    np.nan,
)

# Proved Undeveloped Properties to Assets Q
df['proved_undeveloped_properties_to_assets_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['bal_oil_gas_properties_proved_undeveloped'], df['bal_total_assets']),
    np.nan,
)

# Unproved Properties to Assets Q
df['unproved_properties_to_assets_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['bal_oil_gas_properties_unproved'], df['bal_total_assets']),
    np.nan,
)

# Refining Assets Intensity Q
df['refining_assets_intensity_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['bal_refining_assets_gross'], df['bal_total_assets']),
    np.nan,
)

# Accumulated Depletion Ratio Q
df['accumulated_depletion_ratio_q'] = np.where(
    df['sector'] == 'Energy',
    safe_div_positive(df['bal_accumulated_depletion_depreciation'], ( df['bal_oil_gas_properties_proved_developed'] + df['bal_oil_gas_properties_proved_undeveloped'] + df['bal_oil_gas_properties_unproved'] )),
    np.nan,
)

columns_to_drop = ['shr_shares_change_percentage', 'shr_shares_change_vs_prior_period']
df.drop(columns=columns_to_drop, inplace=True)

df = df.copy() # Just to be safe with the SettingWithCopyWarning appearing from pandas

In [10]:
print("Here are my columns now after feature engineering: ")
for i, col in enumerate(df.columns, start=1): # Looked up this function; allows me to get around the limitations of .columns not displaying every column name
    print(i, col)

Here are my columns now after feature engineering: 
1 ticker
2 fiscal_year
3 fiscal_period
4 period_end_date
5 inc_currency
6 inc_revenue
7 inc_revenue_product
8 inc_revenue_services
9 inc_cost_of_revenue
10 inc_gross_profit
11 inc_operating_expenses
12 inc_research_development
13 inc_selling_general_administrative
14 inc_selling_marketing_expenses
15 inc_general_admin_expenses
16 inc_depreciation_amortization
17 inc_depreciation
18 inc_amortization_of_intangibles
19 inc_stock_based_compensation
20 inc_operating_income
21 inc_interest_expense
22 inc_interest_income
23 inc_other_income_expense
24 inc_income_before_taxes
25 inc_income_tax_expense
26 inc_net_income_continuing_operations
27 inc_net_income_discontinued_operations
28 inc_net_income
29 inc_basic_shares_outstanding
30 inc_diluted_shares_outstanding
31 inc_weighted_average_basic_shares
32 inc_weighted_average_diluted_shares
33 inc_basic_eps
34 inc_diluted_eps
35 inc_dividends_per_share
36 inc_comprehensive_income
37 inc_other_c

In [11]:
# Remove some logistical columns lefterover from 'Building the Dataset' 
df.drop(columns=['buyback_quarter_api_flag', 'buyback_quarter_share_reduction_flag', "sic_code"], errors='ignore', inplace=True)



# Split the df into pre-sector and post-sector specific feature engineering
df_left = df.loc[:, :'prepaid_expenses_to_assets_q'] # Sector agnostic ratios
df_right = df.loc[:, 'fuel_cost_ratio_q':] # Sector specific ratios

# Fill the NaNs for df_right (sector specific) with 0 as these aren't randomly missing values as they genuinely don't apply to those sectors
df_right.fillna(0, inplace=True)

# display(df_left)
# display(df_right)

df = pd.concat([df_left, df_right], axis=1)

# Move predicted variable to end of the dataframe
df['buyback_next_quarter'] = df.pop('buyback_next_quarter')

df.reset_index(inplace=True)

df

,index,ticker,fiscal_year,fiscal_period,period_end_date,inc_currency,inc_revenue,inc_revenue_product,inc_revenue_services,inc_cost_of_revenue,...,ebitdax_margin_q,finding_development_burden_q,commodity_derivative_impact_q,royalty_burden_q,proved_developed_properties_to_assets_q,proved_undeveloped_properties_to_assets_q,unproved_properties_to_assets_q,refining_assets_intensity_q,accumulated_depletion_ratio_q,buyback_next_quarter
0,0,A,2021,Q1,2021-01-31,USD,1.548000e+09,1.172000e+09,376000000.0,710000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,1,A,2021,Q2,2021-04-30,USD,1.525000e+09,1.150000e+09,375000000.0,708000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,2,A,2021,Q3,2021-07-31,USD,1.586000e+09,1.188000e+09,398000000.0,734000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,3,A,2021,Q4,2021-10-31,USD,1.660000e+09,1.246000e+09,414000000.0,760000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,4,A,2022,Q1,2022-01-31,USD,1.674000e+09,1.263000e+09,411000000.0,764000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17314,17314,ZTS,2024,Q3,2024-09-30,USD,2.388000e+09,2.367000e+09,21000000.0,701000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
17315,17315,ZTS,2024,Q4,2024-12-31,USD,2.317000e+09,2.283440e+09,33560000.0,707000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
17316,17316,ZTS,2025,Q1,2025-03-31,USD,2.220000e+09,NaN,NaN,622000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
17317,17317,ZTS,2025,Q2,2025-06-30,USD,2.460000e+09,NaN,NaN,649000000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [12]:
# Export
df.to_csv('df.csv')